# DINOv3 Her Oturumda Başlangıç

Bu notebook, PC'ye kaydettiğiniz eğitilmiş segmentation head'i geri yükler.
Yeniden eğitim yapmaz. Önceki notebook'un değişkenlerine ihtiyaç duymaz.

Her yeni Colab oturumunda T4 GPU seçin ve **1–5 hücrelerini sırayla çalıştırın**.
2. hücrede önceki deney ZIP'inizi veya `head_epoch_05.pth` dosyanızı yükleyin.
ZIP, önceki Oracle ve entropy kayıtlarını da geri getirebilir.
6. hücre tam baseline değerlendirmesidir; 7. hücre sonuçları PC'ye indirir.
Sonraki deney hücrelerini 5. hücreden sonra ekleyebilirsiniz.

Referans: yeni baseline 224 × 224 VOC2012 validation mIoU **0.7104**.
Oracle 50 görüntüde orijinal çözünürlük mIoU **0.4883 → 0.6180**.
Bu iki skor farklı değerlendirme protokollerine aittir.


## 1 Bağımlılıklar

Önceki çalışan Colab oturumundaki Transformers ve Hub sürümlerini kullanır.
PyTorch ve torchvision, Colab ortamından kullanılır ve sürümleri kaydedilir.
Colab yeniden başlatma isterse yeniden başlatıp 1. hücreden devam edin.


In [ ]:
%pip install -q "transformers==5.17.0" "huggingface_hub==1.33.0"


## 2 PC'deki checkpoint paketini yükle

Tek ZIP veya tek PTH seçin. ZIP'te tam bir `head_epoch_05.pth` bulunmalıdır.
Drive bağlantısı gerekmez. GPU mevcut değilse burada durulur.


In [ ]:
import json
import math
import random
import shutil
import zipfile
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from google.colab import files

assert torch.cuda.is_available(), "Çalışma zamanı türünü T4 GPU olarak seçin."
device = torch.device("cuda")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

session_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
session_dir = Path("/content/DINOv3_sessions") / session_id
session_dir.mkdir(parents=True, exist_ok=False)

uploaded = files.upload()
assert len(uploaded) == 1, "Tek bir ZIP veya PTH yükleyin."
upload_name, upload_data = next(iter(uploaded.items()))
upload_path = session_dir / Path(upload_name).name
upload_path.write_bytes(upload_data)

if upload_path.suffix.lower() == ".zip":
    extract_dir = session_dir / "recovered"
    extract_dir.mkdir()
    with zipfile.ZipFile(upload_path) as archive:
        for member in archive.infolist():
            destination = (extract_dir / member.filename).resolve()
            if not destination.is_relative_to(extract_dir.resolve()):
                raise ValueError("ZIP içinde geçersiz dosya yolu.")
        archive.extractall(extract_dir)
    candidates = list(extract_dir.rglob("head_epoch_05.pth"))
    assert len(candidates) == 1, "ZIP içinde tek bir head_epoch_05.pth olmalı."
    checkpoint_path = candidates[0]
elif upload_path.suffix.lower() == ".pth":
    checkpoint_path = upload_path
else:
    raise ValueError("ZIP veya PTH dosyası yükleyin.")

checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
assert "head_state_dict" in checkpoint and "config" in checkpoint
config = checkpoint["config"]
model_name = config["model_name"]
revision = config["revision"]
assert model_name == "facebook/dinov3-vits16-pretrain-lvd1689m"
assert config["input_size"] == 224 and config["num_classes"] == 21
run_dir = checkpoint_path.parent
(run_dir / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
print("GPU:", torch.cuda.get_device_name(0))
print("Checkpoint:", checkpoint_path)
print("Kaydedilen epoch:", checkpoint["epoch"])
print("Çalışma klasörü:", run_dir)


## 3 Backbone ve eğitilmiş head'i geri yükle

Hugging Face girişini tamamlayın. Backbone sabit tutulur; head dosyadan yüklenir.
Bu hücre yeniden çalıştırılırsa aynı kaydedilmiş head yüklenir, eğitim başlamaz.


In [ ]:
import transformers
import torchvision
from huggingface_hub import login
from transformers import AutoImageProcessor, AutoModel

login()
processor = AutoImageProcessor.from_pretrained(model_name, revision=revision)
model = AutoModel.from_pretrained(model_name, revision=revision).to(device)
model.requires_grad_(False)
model.eval()

class DinoSegmentationModel(nn.Module):
    def __init__(self, backbone, num_classes=21):
        super().__init__()
        self.backbone = backbone
        self.backbone.requires_grad_(False)
        self.backbone.eval()
        self.segmentation_head = nn.Conv2d(backbone.config.hidden_size, num_classes, 1)

    def train(self, mode=True):
        super().train(mode)
        self.backbone.eval()
        return self

    def forward(self, pixel_values):
        batch, _, height, width = pixel_values.shape
        patch_size = self.backbone.config.patch_size
        with torch.no_grad():
            hidden = self.backbone(pixel_values=pixel_values).last_hidden_state
        patches = hidden[:, 1 + self.backbone.config.num_register_tokens:, :]
        gh, gw = height // patch_size, width // patch_size
        assert patches.shape[1] == gh * gw
        features = patches.transpose(1, 2).reshape(batch, -1, gh, gw)
        logits = self.segmentation_head(features.to(self.segmentation_head.weight.dtype))
        return F.interpolate(logits, size=(height, width), mode="bilinear", align_corners=False)

restored_model = DinoSegmentationModel(model).to(device)
restored_model.segmentation_head.load_state_dict(checkpoint["head_state_dict"], strict=True)
restored_model.eval()
seg_model = restored_model
assert all(not p.requires_grad for p in model.parameters())
assert (model.config.hidden_size, model.config.patch_size,
        model.config.num_register_tokens) == (384, 16, 4)
for key, value in restored_model.segmentation_head.state_dict().items():
    assert torch.equal(value.cpu(), checkpoint["head_state_dict"][key])
environment = {"torch": str(torch.__version__), "torchvision": torchvision.__version__,
               "transformers": transformers.__version__, "gpu": torch.cuda.get_device_name(0)}
(run_dir / f"environment_{session_id}.json").write_text(json.dumps(environment, indent=2))
print("Backbone ve eğitilmiş head yüklendi. Yeniden eğitim yapılmadı.")
print(environment)


## 4 VOC verisi ve ortak fonksiyonlar

VOC dosyaları geçici runtime'da yoksa yaklaşık 2 GB indirilir.
Maskeler nearest interpolation ile dönüştürülür; 255 etiketi metriklerden çıkarılır.
DataLoader `num_workers=0` kullanır. Orijinal görüntüler crop deneyleri için korunur.


In [ ]:
import torchvision.transforms.functional as TF
from torchvision.datasets import VOCSegmentation
from torchvision.transforms import InterpolationMode
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

class VOCTransform:
    def __init__(self, processor, size=224):
        self.size = size
        self.mean, self.std = processor.image_mean, processor.image_std

    def __call__(self, image, mask):
        image = TF.resize(image.convert("RGB"), [self.size, self.size],
                          interpolation=InterpolationMode.BICUBIC)
        mask = TF.resize(mask, [self.size, self.size], interpolation=InterpolationMode.NEAREST)
        image = TF.normalize(TF.to_tensor(image), self.mean, self.std)
        return image, torch.from_numpy(np.array(mask, dtype=np.int64, copy=True))

transform = VOCTransform(processor)
val_dataset = VOCSegmentation("/content/VOC", year="2012", image_set="val",
                              download=True, transforms=transform)
train_dataset = VOCSegmentation("/content/VOC", year="2012", image_set="train",
                                download=False, transforms=transform)
raw_val_dataset = VOCSegmentation("/content/VOC", year="2012", image_set="val", download=False)
train_ids = {Path(p).stem for p in train_dataset.images}
val_ids = {Path(p).stem for p in val_dataset.images}
assert train_ids.isdisjoint(val_ids)
assert len(val_dataset) == 1449 and len(train_dataset) == 1464
for split, ids in (("train", train_ids), ("val", val_ids)):
    previous = run_dir / f"{split}_ids.txt"
    if previous.exists():
        assert set(previous.read_text().splitlines()) == ids, f"{split} split değişmiş."

val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0, pin_memory=True)
class_names = ["background", "aeroplane", "bicycle", "bird", "boat", "bottle", "bus", "car",
               "cat", "chair", "cow", "diningtable", "dog", "horse", "motorbike", "person",
               "pottedplant", "sheep", "sofa", "train", "tvmonitor"]

@torch.inference_mode()
def predict_logits(rgb):
    image = TF.resize(rgb.convert("RGB"), [224, 224], interpolation=InterpolationMode.BICUBIC)
    image = TF.normalize(TF.to_tensor(image), processor.image_mean, processor.image_std)
    return restored_model(image.unsqueeze(0).to(device)).float().cpu()

def normalized_entropy(logits):
    probabilities = logits.float().softmax(1)
    return (-(probabilities * probabilities.clamp_min(1e-12).log()).sum(1)
            / math.log(logits.shape[1])).clamp(0, 1)

def confusion_for(prediction, target):
    valid = target != 255
    gt, pred = target[valid].cpu(), prediction[valid].cpu()
    assert torch.all((gt >= 0) & (gt < 21))
    assert torch.all((pred >= 0) & (pred < 21))
    return torch.bincount(gt * 21 + pred, minlength=441).reshape(21, 21)

def scores_from_confusion(confusion):
    intersection = confusion.diag().double()
    union = confusion.sum(0) + confusion.sum(1) - confusion.diag()
    present = union > 0
    scores = torch.full((21,), float("nan"), dtype=torch.float64)
    scores[present] = intersection[present] / union[present]
    return scores, present

def image_miou(pred, target):
    scores, present = scores_from_confusion(confusion_for(pred, target))
    return scores[present].mean().item()

def target_iou(pred, target, class_id):
    valid = target != 255
    predicted, actual = (pred == class_id) & valid, (target == class_id) & valid
    union = (predicted | actual).sum().item()
    return (predicted & actual).sum().item() / union if union else float("nan")

print("VOC hazır: train=1464, validation=1449. Ortak fonksiyonlar hazır.")


## 5 Tek görüntü kontrolü

Her oturumda bu hücreyi çalıştırın. Sadece inference yapılır.
GT içindeki ignore pikselleri segmentation görselinde siyah gösterilir.
Bu hücreden sonra selector ve diğer deneylere geçilebilir.


In [ ]:
import matplotlib.pyplot as plt

IMAGE_INDEX = 9
image, mask = raw_val_dataset[IMAGE_INDEX]
logits = predict_logits(image)
assert logits.shape == (1, 21, 224, 224) and torch.isfinite(logits).all()
pred = logits.argmax(1)[0]
gt = transform(image, mask)[1]
entropy = normalized_entropy(logits)[0]
assert torch.isfinite(entropy).all()
assert not restored_model.backbone.training

palette = plt.get_cmap("tab20").copy()
palette.set_bad("black")
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(image); axes[0].set_title("Original")
axes[1].imshow(np.ma.masked_equal(gt.numpy(), 255), cmap=palette, vmin=0, vmax=20)
axes[1].set_title("GT — ignore black")
axes[2].imshow(pred.numpy(), cmap=palette, vmin=0, vmax=20); axes[2].set_title("Prediction")
im = axes[3].imshow(entropy.numpy(), cmap="inferno", vmin=0, vmax=1)
axes[3].set_title("Entropy"); fig.colorbar(im, ax=axes[3], fraction=0.046)
for ax in axes: ax.axis("off")
plt.tight_layout()
fig.savefig(run_dir / f"startup_preview_{session_id}.png", dpi=150)
plt.show()
print("Başlangıç kontrolü geçti.")
print("Örnek görüntü mIoU (224 x 224):", round(image_miou(pred, gt), 4))
print("Sonraki deneyler için ortam hazır.")


## 6 Tam baseline değerlendirmesi isteğe bağlı

Checkpoint'i ilk kez bu notebook'ta açtığınızda çalıştırmanız önerilir.
1449 görüntü, 224 × 224 değerlendirme, background dahil, ignore=255.
Kaydedilen 0.7104 ile aynı protokoldür. Sonuç ölçülür, eşleşmesi zorlanmaz.


In [ ]:
confusion = torch.zeros((21, 21), dtype=torch.int64)
evaluated_images = 0
with torch.inference_mode():
    for images, masks in tqdm(val_loader, desc="Validation"):
        predictions = restored_model(images.to(device)).argmax(1).cpu()
        confusion += confusion_for(predictions, masks)
        evaluated_images += images.shape[0]
assert evaluated_images == 1449
class_iou, present = scores_from_confusion(confusion)
miou = class_iou[present].mean().item()
results = {"checkpoint": checkpoint_path.name, "validation_images": evaluated_images,
           "evaluation_resolution": [224, 224], "ignore_index": 255, "background_included": True,
           "absent_class_policy": "exclude classes with zero union", "miou": miou,
           "class_iou": {name: float(class_iou[i]) if present[i] else None
                         for i, name in enumerate(class_names)},
           "confusion_matrix": confusion.tolist()}
result_path = run_dir / f"validation_results_{session_id}.json"
result_path.write_text(json.dumps(results, indent=2, allow_nan=False), encoding="utf-8")
print(f"Validation mIoU: {miou:.4f}")
print("Önceki ölçüm: 0.7104; fark:", round((miou - 0.7104) * 100, 4), "yüzde puan")
print("Kaydedildi:", result_path)


## 7 Oturum sonunda PC'ye indir

Deneyleriniz bittikten sonra çalıştırın. Checkpoint ve aynı klasöre kaydedilen
deney sonuçları ZIP'e eklenir. Sonraki oturumda bu güncel ZIP'i yükleyin.
Colab oturumunu kapatmadan indirme işleminin tamamlandığını kontrol edin.
Notebook'u da Colab Dosya menüsünden kaydedin veya indirin; checkpoint ZIP'i kod içermez.


In [ ]:
export_path = shutil.make_archive(
    str(Path("/content") / f"DINOv3_baseline_{session_id}"), "zip",
    root_dir=str(run_dir.parent), base_dir=run_dir.name,
)
files.download(export_path)
print("PC'ye indirilecek güncel paket:", export_path)
